# Tarea C6 — Evaluación + Exportar ONNX
**ECG Rhythm Analyzer | Gabriela — Modelo CNN**  
**Rama:** `feature/model-training`

## Objetivo
1. Evaluar el modelo entrenado en el test set (datos nunca vistos)
2. Calcular métricas: accuracy, sensibilidad, especificidad, AUC-ROC
3. Generar matriz de confusión y curva ROC
4. Analizar los ECGs peor clasificados
5. Exportar el modelo a formato ONNX para Cristina


## PASO 0 — Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
import torch.nn as nn
import numpy as np
import os
import matplotlib.pyplot as plt
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

DEVICE         = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BASE_DIR       = '/content/ecg_images2/ecg_images'
CHECKPOINT_DIR = '/content/drive/MyDrive/ecg_checkpoints'
IMG_SIZE       = 224
BATCH_SIZE     = 32

print(f'Dispositivo: {DEVICE}')

# Descomprimir imágenes si no existen
if not os.path.exists(BASE_DIR):
    print('Descomprimiendo imágenes...')
    os.system('unzip /content/drive/MyDrive/ecg_images_dataset.zip -d /content/ecg_images2')
    print('Listo.')

## PASO 1 — Cargar el modelo entrenado

In [ ]:
# Cargar EfficientNet-B0 con los pesos del mejor modelo de Fase 2
model = models.efficientnet_b0(weights=None)  # Sin pesos de ImageNet, vamos a cargar los nuestros
model.classifier[1] = nn.Linear(1280, 1)

checkpoint_path = os.path.join(CHECKPOINT_DIR, 'modelo_fase2.pt')
model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))
model = model.to(DEVICE)
model.eval()

print(f'✅ Modelo cargado desde: {checkpoint_path}')

## PASO 2 — Data Loader del Test Set

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

test_dataset = datasets.ImageFolder(os.path.join(BASE_DIR, 'test'), transform=test_transform)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'Test set: {len(test_dataset)} imágenes')
print(f'Clases: {test_dataset.classes}')

## PASO 3 — Predicciones en el Test Set

In [ ]:
all_labels = []
all_probs  = []
all_preds  = []

print('Evaluando en test set...')
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(DEVICE)
        outputs = model(images).squeeze()
        probs   = torch.sigmoid(outputs).cpu().numpy()
        preds   = (probs > 0.5).astype(int)

        all_labels.extend(labels.numpy())
        all_probs.extend(probs)
        all_preds.extend(preds)

all_labels = np.array(all_labels)
all_probs  = np.array(all_probs)
all_preds  = np.array(all_preds)

print(f'✅ Predicciones completas: {len(all_labels)} imágenes evaluadas.')

## PASO 4 — Métricas

In [ ]:
from sklearn.metrics import (
    accuracy_score, recall_score, roc_auc_score,
    confusion_matrix, classification_report
)

# Calcular métricas
accuracy      = accuracy_score(all_labels, all_preds)
sensibilidad  = recall_score(all_labels, all_preds, pos_label=1)   # recall para arritmia
especificidad = recall_score(all_labels, all_preds, pos_label=0)   # recall para normal
auc_roc       = roc_auc_score(all_labels, all_probs)

print('=' * 45)
print('MÉTRICAS EN TEST SET (datos nunca vistos)')
print('=' * 45)
print(f'Accuracy:       {accuracy*100:.2f}%')
print(f'Sensibilidad:   {sensibilidad*100:.2f}%  ← cuántas arritmias detectó correctamente')
print(f'Especificidad:  {especificidad*100:.2f}%  ← cuántos normales identificó correctamente')
print(f'AUC-ROC:        {auc_roc:.4f}  ← entre 0.5 (azar) y 1.0 (perfecto)')
print()
print('Reporte completo:')
print(classification_report(all_labels, all_preds, target_names=['normal', 'arritmia']))

## PASO 5 — Matriz de Confusión y Curva ROC

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Evaluación del Modelo — Test Set ECG', fontsize=13)

# ─── Matriz de confusión ─────────────────────────────────────
cm = confusion_matrix(all_labels, all_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Normal', 'Arritmia'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Matriz de Confusión', fontsize=11)

# Explicación de la matriz
tn, fp, fn, tp = cm.ravel()
print('=== Matriz de Confusión ===')
print(f'Verdaderos Negativos (Normal → Normal):     {tn}')
print(f'Falsos Positivos (Normal → Arritmia):       {fp}  ← alarmas falsas')
print(f'Falsos Negativos (Arritmia → Normal):       {fn}  ← arritmias no detectadas')
print(f'Verdaderos Positivos (Arritmia → Arritmia): {tp}')

# ─── Curva ROC ───────────────────────────────────────────────
RocCurveDisplay.from_predictions(
    all_labels, all_probs,
    name=f'EfficientNet-B0 (AUC={auc_roc:.3f})',
    ax=axes[1]
)
axes[1].plot([0,1], [0,1], 'k--', label='Azar (AUC=0.5)')
axes[1].set_title('Curva ROC', fontsize=11)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(CHECKPOINT_DIR, 'evaluacion_test.png'), dpi=100, bbox_inches='tight')
plt.show()
print('\nGráfica guardada en Drive.')

## PASO 6 — Análisis de errores
Identificar los 20 ECGs peor clasificados (mayor confianza incorrecta)

In [ ]:
from PIL import Image

# Encontrar predicciones incorrectas con mayor confianza
incorrectos = []
for i in range(len(all_labels)):
    if all_preds[i] != all_labels[i]:
        confianza = all_probs[i] if all_preds[i] == 1 else 1 - all_probs[i]
        incorrectos.append({
            'idx': i,
            'label_real': all_labels[i],
            'pred': all_preds[i],
            'prob': all_probs[i],
            'confianza_incorrecta': confianza
        })

# Ordenar por confianza incorrecta (peores primero)
incorrectos.sort(key=lambda x: x['confianza_incorrecta'], reverse=True)
peores_20 = incorrectos[:20]

print(f'Total de predicciones incorrectas: {len(incorrectos)} de {len(all_labels)}')
print(f'Tasa de error: {len(incorrectos)/len(all_labels)*100:.2f}%')
print()
print('Top 5 peores (mayor confianza incorrecta):')
clases = ['normal', 'arritmia']
for e in peores_20[:5]:
    print(f'  idx={e["idx"]} | Real: {clases[e["label_real"]]} | Pred: {clases[e["pred"]]} | Prob arritmia: {e["prob"]:.3f}')

In [ ]:
# Mostrar las imágenes de los 12 peores casos
fig, axes = plt.subplots(3, 4, figsize=(14, 10))
fig.suptitle('Los 12 ECGs peor clasificados (mayor confianza incorrecta)', fontsize=12)

# Obtener rutas de imágenes del test set
test_img_paths = [path for path, _ in test_dataset.imgs]

for i, error in enumerate(peores_20[:12]):
    ax  = axes[i // 4][i % 4]
    idx = error['idx']

    img = Image.open(test_img_paths[idx])
    ax.imshow(img)

    real = clases[error['label_real']]
    pred = clases[error['pred']]
    conf = error['confianza_incorrecta']
    ax.set_title(f'Real: {real}\nPred: {pred} ({conf:.0%})', fontsize=8, color='red')
    ax.axis('off')

plt.tight_layout()
plt.savefig(os.path.join(CHECKPOINT_DIR, 'peores_clasificaciones.png'), dpi=100, bbox_inches='tight')
plt.show()
print('¿Ves algún patrón en los errores? (señales con ruido, imágenes borrosas, casos límite?)')

## PASO 7 — Exportar modelo a ONNX para Cristina
ONNX es un formato universal que permite usar el modelo en cualquier lenguaje/plataforma, incluyendo la app de Streamlit de Cristina.

In [ ]:
import torch.onnx

# Cargar el mejor modelo
model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))
model.eval()

# Input dummy con la forma correcta: (1, 3, 224, 224)
# 1=una imagen, 3=canales RGB, 224x224=tamaño
dummy_input = torch.randn(1, 3, 224, 224).to(DEVICE)

# Ruta de salida del ONNX
onnx_path = os.path.join(CHECKPOINT_DIR, 'ecg_model.onnx')

# Exportar
torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    export_params=True,
    opset_version=11,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={
        'input':  {0: 'batch_size'},
        'output': {0: 'batch_size'}
    }
)

size_mb = os.path.getsize(onnx_path) / (1024 * 1024)
print(f'✅ Modelo exportado a ONNX')
print(f'   Ruta: {onnx_path}')
print(f'   Tamaño: {size_mb:.1f} MB')

In [ ]:
# Verificar que el ONNX funciona correctamente
!pip install onnxruntime -q
import onnxruntime as ort

# Cargar el modelo ONNX
session = ort.InferenceSession(onnx_path)

# Correr una predicción de prueba
test_input = np.random.randn(1, 3, 224, 224).astype(np.float32)
result     = session.run(None, {'input': test_input})
prob       = 1 / (1 + np.exp(-result[0][0][0]))  # sigmoid

print('=== Verificación del modelo ONNX ===')
print(f'Input shape:  {test_input.shape}')
print(f'Output logit: {result[0][0][0]:.4f}')
print(f'Probabilidad (arritmia): {prob:.4f}')
print()
print('✅ El modelo ONNX funciona correctamente.')
print('   Cristina puede usar este archivo en su app de Streamlit.')

## PASO 8 — Resumen final para el equipo

In [ ]:
print(f"""
╔══════════════════════════════════════════════════╗
║     RESUMEN FINAL — GABRIELA (MODELO CNN)        ║
╠══════════════════════════════════════════════════╣
║  MODELO: EfficientNet-B0 (transfer learning)     ║
║  DATOS:  {len(test_dataset)} imágenes en test set               ║
╠══════════════════════════════════════════════════╣
║  MÉTRICAS EN TEST SET                            ║
║  Accuracy:      {accuracy*100:.2f}%                        ║
║  Sensibilidad:  {sensibilidad*100:.2f}%  (detección arritmia)  ║
║  Especificidad: {especificidad*100:.2f}%  (detección normal)   ║
║  AUC-ROC:       {auc_roc:.4f}                        ║
╠══════════════════════════════════════════════════╣
║  ENTREGABLE PARA CRISTINA                        ║
║  📦 ecg_model.onnx ({size_mb:.1f} MB)                  ║
║  📁 {onnx_path[:45]}  ║
╚══════════════════════════════════════════════════╝
""")

## ✅ Semana 3 Completada

**Pasos finales:**
1. Descargar `ecg_model.onnx` desde Drive y subirlo al repositorio GitHub en la carpeta `model/`
2. Avisar a Cristina que el modelo está listo
3. Subir C5 y C6 a GitHub en la rama `feature/model-training`